# Zchezz — Pipeline de Self-Play em Shards (~10M Posições) + Treino na GPU
### Otimizado para Google Colab Pro com Persistência Atômica no Google Drive

Este notebook suporta dois perfis de motor independentes:
- **Conta 1 (`PROFILE = 'v507'`):** Geração nativa em C multithread in-process (NNU4).
- **Conta 2 (`PROFILE = 'v331'`):** Geração via runner UCI de processos persistentes (NNU3).

> [!IMPORTANT]
> **Configuração no Colab Pro:**
> 1. **Fase 1 e 2 (Calibração e Geração):** Runtime **CPU** com perfil **High-RAM / High-CPU** (8 a 12 vCPUs).
> 2. **Fase 3 (Treinamento):** Runtime **GPU (T4 ou A100)** com perfil **High-RAM**.
> 3. Se disponível na sua assinatura Pro/Pro+, ative **Execução em segundo plano (Background execution)**.

In [ ]:
# ==================== SELEÇÃO DE PERFIL E CONTA ====================
# Cada conta Colab DEVE usar valores específicos aqui:
# Conta 1: PROFILE = 'v507', ACCOUNT_ID = 1
# Conta 2: PROFILE = 'v331', ACCOUNT_ID = 2
PROFILE = 'v507'          # 'v507' | 'v331'
ACCOUNT_ID = 1            # 1 | 2 (determina a faixa de seeds dos shards)
TARGET_POSITIONS = 5000000# Meta de posições por conta (5M por conta = 10M total)
GAMES_PER_SHARD = 3000    # Partidas por shard (~20 a 30 min por shard; par)
REPO_URL = 'https://github.com/gitzambrano/zchezz.git'
REPO_REF = 'claude/selfplay-10m-colab-pipeline-5efe32'  # branch ou commit com colab/
# ====================================================================
print(f"[OK] Configurado para Conta {ACCOUNT_ID} - Perfil: {PROFILE} (Meta: {TARGET_POSITIONS:,} posições)")

In [ ]:
# 1. Diagnóstico de Hardware e Verificação de Instruções AVX2
!echo '=== CPU Info ==='
!lscpu | grep -E 'Model name|CPU\(s\):|Thread|MHz|Flags'

import subprocess
flags = subprocess.getoutput('lscpu | grep -i flags')
if 'avx2' not in flags.lower():
    raise RuntimeError('ERRO CRÍTICO: Esta máquina Colab não possui AVX2. Altere o tipo de ambiente para uma CPU moderna.')
print('\n[OK] Suporte a AVX2 confirmado.')

!echo '\n=== Memória RAM ==='
!free -h
!echo '\n=== GPU (se habilitada) ==='
!nvidia-smi || echo "(Modo CPU ativo para geração - correto)"

In [ ]:
# 2. Conectar ao Google Drive para persistência dos shards
from google.colab import drive
import os

drive.mount("/content/drive")

DRIVE_DIR = f"/content/drive/MyDrive/zchezz_data/selfplay_{PROFILE}"
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f"[OK] Diretório do Drive montado: {DRIVE_DIR}")

In [ ]:
# 3. Clonar o repositório no REPO_REF e compilar o motor
# O Makefile nomeia os binários com .exe também no Linux.
import os

if not os.path.exists("/content/Zchezz/.git"):
    !git clone {REPO_URL} /content/Zchezz
%cd /content/Zchezz
!git fetch --quiet origin
!git checkout --quiet {REPO_REF} && git reset --quiet --hard origin/{REPO_REF} 2>/dev/null || git checkout --quiet {REPO_REF}
!git log --oneline -1
assert os.path.exists("colab/run_selfplay_colab.sh"), "REPO_REF não contém colab/: confira o branch"

if PROFILE == 'v507':
    !make -C engine/build TOOLS_ENGINE=v507 STATIC_FLAG="" selfplay
    !./engine/build/selfplay.exe --help 2>&1 | head -n 5
elif PROFILE == 'v331':
    !make -C engine/build ENGINE=v331 STATIC_FLAG="" native
    !printf 'uci
quit
' | ./engine/c/zchezz_v331/zchezz.exe | head -n 5


## 4. Fase 1: Calibração Obrigatória
Antes de gerar milhões de posições, medimos:
1. Amostras úteis por partida (lances de abertura excluídos);
2. Vazão real (posições/segundo);
3. ETA para a meta.

Para v507 a célula testa vários `--nodes`. Um smoke local com 5.000 nós deu cerca de 370 posições/s por thread, então prefira o maior valor que caiba no orçamento de horas (rótulos mais profundos). Para v331 a célula testa `--movetime`.


In [ ]:
# Fase 1: Calibração
import time, sys, glob
sys.path.insert(0, '/content/Zchezz/train')
from dataset import MultiShardSelfplay

CALIB_DIR = '/content/calib_test'
results = []

if PROFILE == 'v507':
    CALIB_GAMES = 200
    for nodes in (5000, 10000, 20000, 40000):
        !rm -rf {CALIB_DIR} && mkdir -p {CALIB_DIR}
        out = f'{CALIB_DIR}/calib.bin'
        t0 = time.time()
        !./engine/build/selfplay.exe --out {out} --nnue engine/c/zchezz_v507/nnue_weights.bin             --games {CALIB_GAMES} --threads 0 --movetime 0 --nodes {nodes}             --multipv 4 --temperature 1.0 --temp-plies 24 --temp-final 0             --no-same-opening-twice --opening-mode random --random-plies 8 --seed 999 2>&1 | tail -n 1
        results.append((f'nodes={nodes}', len(MultiShardSelfplay([out])), time.time() - t0))
else:
    CALIB_GAMES = 50   # jogos; o runner recebe aberturas = jogos / 2
    for movetime in (20, 50, 100):
        !rm -rf {CALIB_DIR} && mkdir -p {CALIB_DIR}
        t0 = time.time()
        !python3 tests/run_selfplay.py --profile v331 --bin --no-epd --no-pgn --no-opening-in-bin             --no-same-opening-twice --opening-mode random --random-plies 8 --games {CALIB_GAMES // 2}             --concurrency $(nproc) --movetime {movetime} --seed 999 --results-dir {CALIB_DIR} > /dev/null
        results.append((f'movetime={movetime}ms', len(MultiShardSelfplay.from_glob(f'{CALIB_DIR}/*.bin')), time.time() - t0))

print('=' * 72)
print(f'CALIBRAÇÃO {PROFILE} ({CALIB_GAMES} jogos por ponto)')
print(f'{"config":<18}{"pos/jogo":>10}{"pos/s":>10}{"ETA meta (h)":>16}')
for name, samples, elapsed in results:
    pps = samples / elapsed
    print(f'{name:<18}{samples / CALIB_GAMES:>10.1f}{pps:>10.1f}{TARGET_POSITIONS / pps / 3600:>16.2f}')
print('=' * 72)
!rm -rf {CALIB_DIR}


## 5. Fase 2: Geração de Shards com Persistência Atômica no Drive
Defina o valor de `CALIBRATED_NODES` (para v507) obtido na calibração acima e execute o gerador de shards.
O script grava os shards localmente em `/content/zchezz_shards`, valida a integridade com `MultiShardSelfplay`, gera o sidecar `.json` de metadados e move atomicamente para o Google Drive (`.tmp` -> `mv`).

Se a sessão for reiniciada, basta reexecutar esta célula: todos os shards já presentes no Drive são detectados e pulados automaticamente.

In [ ]:
# Fase 2: Execução do Shard Runner
# Ajuste os parâmetros calibrados conforme os resultados da Fase 1:
CALIBRATED_NODES = 5000   # Obrigatório > 0 para v507
CALIBRATED_MOVETIME = 50 # Utilizado para v331

!PROFILE={PROFILE} \
 ACCOUNT_ID={ACCOUNT_ID} \
 TARGET_POSITIONS={TARGET_POSITIONS} \
 GAMES_PER_SHARD={GAMES_PER_SHARD} \
 NODES={CALIBRATED_NODES} \
 MOVETIME={CALIBRATED_MOVETIME} \
 THREADS=0 \
 bash colab/run_selfplay_colab.sh

## 6. Validação Completa do Dataset no Drive
Carrega todos os shards do Google Drive via `MultiShardSelfplay.from_glob(...)`, exibe a distribuição de resultados e avaliações `eval_cp`, e executa o validador de invariantes `tests/test_selfplay_bin.py`.

In [ ]:
# 6. Inspeção e Validação de Invariantes dos Shards no Drive
import glob, os, sys
sys.path.insert(0, '/content/Zchezz/train')
from dataset import MultiShardSelfplay
import numpy as np

pattern = f'{DRIVE_DIR}/sp_{PROFILE}_a{ACCOUNT_ID}_s*.bin'
shards = sorted(glob.glob(pattern))
print(f'Total de shards encontrados: {len(shards)}')

if shards:
    dataset = MultiShardSelfplay.from_glob(pattern)
    total = len(dataset)
    print('=' * 60)
    print(f'Total de posições acumuladas no Drive: {total:,}')
    print('Proveniência:')
    for p in dataset.provenance_summary()[:3]:
        print(' ', p)
    if len(dataset.provenance_summary()) > 3:
        print(f'  ... e mais {len(dataset.provenance_summary()) - 3} shards.')
    print('=' * 60)
    
    # Validação com test_selfplay_bin.py em uma amostra de shards
    sample_shards = shards[:min(5, len(shards))]
    print(f'Rodando test_selfplay_bin.py em {len(sample_shards)} shards...')
    !python3 tests/test_selfplay_bin.py {' '.join(sample_shards)}
else:
    print('Nenhum shard encontrado no Drive ainda.')

## 7. Fase 3: Treinamento da Rede na GPU (Runtime GPU High-RAM)
Quando a meta de posições for atingida:
1. Altere o tipo de ambiente para **GPU** (T4 ou A100) com **High-RAM**. O runtime novo começa vazio: rode de novo a célula de configuração, a de montagem do Drive e a de clone antes desta.
2. Copie os shards do Drive para `/content/shards` (leitura local é ordens de grandeza mais rápida que FUSE do Drive).
3. Crie o symlink do diretório de checkpoints para persistir `latest.pt` direto no Drive.
4. Execute `train/run.py --profile <PROFILE>`.

In [ ]:
# Fase 3: Treinamento PyTorch na GPU
import os

# 1. Criar symlink de checkpoints para o Drive
DRIVE_CKPT = f'/content/drive/MyDrive/zchezz_data/checkpoints/{PROFILE}'
LOCAL_CKPT = f'/content/Zchezz/checkpoints/{PROFILE}'
os.makedirs(DRIVE_CKPT, exist_ok=True)
!rm -rf {LOCAL_CKPT} && ln -s {DRIVE_CKPT} {LOCAL_CKPT}
print(f'[OK] Checkpoints vinculados: {LOCAL_CKPT} -> {DRIVE_CKPT}')

# 2. Copiar shards do Drive para disco local rápido
LOCAL_SHARDS = '/content/shards'
!mkdir -p {LOCAL_SHARDS} && cp -u {DRIVE_DIR}/*.bin {LOCAL_SHARDS}/
!ls -la {LOCAL_SHARDS} | head -n 10

# 3. Exibir configuração do trainer
!python3 train/run.py --profile {PROFILE} --show-config

# 4. Disparo do Treino (meça a 1ª época para calibrar o número total de épocas)
EPOCHS = 10
K_BLEND = 0.75
!python3 train/run.py \
    --profile {PROFILE} \
    --source kind=bin,path={LOCAL_SHARDS}/*.bin,k={K_BLEND} \
    --epochs {EPOCHS} \
    --workers $(nproc)

print('[OK] Treino concluído! O arquivo latest.pt foi salvo no Google Drive.')